In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
!pip install -q langchain_openai==0.3.33 langchain==0.3.27 langchain-community==0.3.24

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.8/450.8 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 99.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.


In [8]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
from langchain import OpenAI, ConversationChain

In [9]:
llm = OpenAI(temperature=0)
conversation = ConversationChain(llm=llm)

In [10]:
response1 = conversation.invoke("Who won the 2018 FIFA world cup?")
response1

{'input': 'Who won the 2018 FIFA world cup?',
 'history': '',
 'response': " The 2018 FIFA World Cup was won by France. They defeated Croatia 4-2 in the final match. This was France's second World Cup victory, with their first being in 1998. The tournament was held in Russia and lasted from June 14 to July 15, 2018. France's star player, Kylian Mbappé, was named the Best Young Player of the tournament."}

In [11]:
response2 = conversation.invoke("Who won the next?")
response2

{'input': 'Who won the next?',
 'history': "Human: Who won the 2018 FIFA world cup?\nAI:  The 2018 FIFA World Cup was won by France. They defeated Croatia 4-2 in the final match. This was France's second World Cup victory, with their first being in 1998. The tournament was held in Russia and lasted from June 14 to July 15, 2018. France's star player, Kylian Mbappé, was named the Best Young Player of the tournament.",
 'response': ' The next FIFA World Cup will be held in 2022 in Qatar. The winner of the 2022 World Cup has not yet been determined, as the qualifying rounds are still ongoing. However, the current top-ranked team in the world is Belgium, followed by France and Brazil.'}

In [12]:
conversation2 = ConversationChain(llm=llm)

In [13]:
response1 = conversation2.invoke("what is my name?")
response1

{'input': 'what is my name?',
 'history': '',
 'response': ' I am sorry, I do not have access to your personal information. I am an AI and do not have the ability to know your name unless you tell me. Is there something else you would like to know?'}

In [14]:
response2 = conversation2.invoke("what is the date today?")
response2

{'input': 'what is the date today?',
 'history': 'Human: what is my name?\nAI:  I am sorry, I do not have access to your personal information. I am an AI and do not have the ability to know your name unless you tell me. Is there something else you would like to know?',
 'response': ' Today is Tuesday, October 12th, 2021. Is there anything else you would like to know?'}

In [15]:
from langchain.tools import Tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.prompts import PromptTemplate

In [16]:
from datetime import datetime
today_tool = Tool(
    name="Date Today",
    func=lambda x:datetime.now().strftime("%Y-%m-%d"),
    description="Returns the current date",
)

In [17]:
template = """You are a helpful assistant who can use tools to answer user's questions.
You can use the Date Today tool to get the current date. Question: {question}"""

prompt = PromptTemplate(
    template=template,
    input_variables=["question"],
)

In [18]:
llm = ChatOpenAI(temperature=0)

agent = initialize_agent(
    tools=[today_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False
)

In [19]:
agent.run(prompt.format(question='what is the date today?'))

'The date today is 2025-12-06.'

In [20]:
agent.run(prompt.format(question='how many days to new year?'))

'There are 26 days left until the new year.'

In [21]:
agent = initialize_agent(
    tools=[today_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

In [22]:
agent.run(prompt.format(question='what is the date today?'))



> Entering new AgentExecutor chain...
I should use the Date Today tool to get the current date.
Action: Date Today
Action Input: 
Observation: 2025-12-06
Thought:I now know the final answer
Final Answer: The date today is 2025-12-06.

> Finished chain.


'The date today is 2025-12-06.'

In [23]:
agent.run(prompt.format(question='what is the current year?'))



> Entering new AgentExecutor chain...
I should use the Date Today tool to get the current year.
Action: Date Today
Action Input: x
Observation: 2025-12-06
Thought:The current year is 2025.
Final Answer: 2025

> Finished chain.


'2025'

In [24]:
agent.run(prompt.format(question='what is the current week?'))



> Entering new AgentExecutor chain...
I should use the Date Today tool to get the current date and then determine the week number.
Action: Date Today
Action Input: 
Observation: 2025-12-06
Thought:Now I need to determine the week number for the current date.
Action: Week Number
Action Input: 2025-12-06
Observation: Week Number is not a valid tool, try one of [Date Today].
Thought:I should try a different approach to determine the current week number.
Action: Calculate Week Number
Action Input: 2025-12-06
Observation: Calculate Week Number is not a valid tool, try one of [Date Today].
Thought:I can simply count the number of weeks that have passed in the current year.
Action: Calculate Weeks Passed
Action Input: 2025-12-06
Observation: Calculate Weeks Passed is not a valid tool, try one of [Date Today].
Thought:I can calculate the week number manually by dividing the day of the year by 7.
Action: Calculate Week Number Manually
Action Input: 2025-12-06
Observation: Calculate Week Numbe

'I cannot determine the current week number using the tools available.'